# Part 6 · Notebook 07 — The volatility surface: SVI, arbitrage checks and SABR

**Sessions:** S7 (Volatility surface) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Write the raw SVI smile and fit it to market IVs.
2. Check a smile for butterfly arbitrage and a surface for calendar arbitrage.
3. Interpolate between expiries in total variance.
4. See how SABR's parameters shape a smile.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()

## 1. Raw SVI

Gatheral's raw SVI describes one expiry's **total implied variance** `w = σ²·T` as a function of log-moneyness `k = ln(K/F)`: `w(k) = a + b·(ρ(k − m) + √((k − m)² + s²))`. `a` sets the level, `b` the slope of the wings, `ρ` the skew, `m` shifts it, `s` rounds the bottom.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def svi_total_var(k, a, b, rho, m, s):
    return ...                                    # ✍️

k = np.linspace(-0.25, 0.15, 9)
mine = svi_total_var(k, *p.TRUE_SVI)
mine = p.check("svi_total_var", mine, p.svi_total_var(k, *p.TRUE_SVI))
np.sqrt(np.asarray(mine) / (30 / 365)).round(4)       # as implied vols

## 2. Fit it to a chain

Take the liquid out-of-the-money quotes, compute IVs off the parity-implied forward (notebook 04), and fit SVI by bounded least squares on total variance. The chain was generated from `p.TRUE_SVI`, so we can see whether the fit recovers it.

In [ ]:
S, T, r, q = 600.0, 30 / 365, 0.045, 0.013
chain = p.liquidity_filter(p.synthetic_chain(S, T, r, q))
both = chain.pivot_table(index="strike", columns="cp", values="mid").dropna()
near = both[(both.index > 570) & (both.index < 630)]
r_imp, F = p.implied_rate_forward(near.index, near[1], near[-1], T)
otm = chain[((chain.cp == -1) & (chain.strike < F)) | ((chain.cp == 1) & (chain.strike >= F))]
iv = np.array([p.implied_vol(m, F, K, T, r_imp, r_imp, cp) for m, K, cp in zip(otm["mid"], otm.strike, otm.cp)])
kk = np.log(otm.strike.to_numpy() / F)
fit = p.fit_svi(kk, iv, T)
display(pd.DataFrame({"true": p.TRUE_SVI, "fitted": fit}, index=["a", "b", "rho", "m", "s"]).round(4))
grid = np.linspace(kk.min(), kk.max(), 200)
plt.plot(kk, iv * 100, "o", ms=4, label="market IV (OTM mids)")
plt.plot(grid, np.sqrt(p.svi_total_var(grid, *fit) / T) * 100, label="SVI fit")
plt.xlabel("k = ln(K/F)"); plt.ylabel("IV, %"); plt.title("A 30-day equity smile"); plt.legend(); plt.show()

## 3. Butterfly arbitrage

A smile implies a risk-neutral density; if the density is negative somewhere, a butterfly spread there has a negative price: free money, or more likely a broken fit. Durrleman's condition `g(k) >= 0` checks it (`p.butterfly_g`). An unconstrained fit with wings that are too steep fails:

In [ ]:
bad = (0.002, 0.2, -0.8, 0.0, 0.01)
k = np.linspace(-0.4, 0.3, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for prm, name in [(p.TRUE_SVI, "fitted-style smile"), (bad, "too-steep wings")]:
    axes[0].plot(k, np.sqrt(p.svi_total_var(k, *prm) / T) * 100, label=name)
    axes[1].plot(k, p.butterfly_g(k, *prm), label=name)
axes[1].axhline(0, color="black", lw=0.8); axes[1].set_ylim(-3, 3)
axes[0].set_title("IV, %"); axes[1].set_title("g(k): must stay >= 0"); axes[1].legend(); plt.tight_layout(); plt.show()
print(f"bad smile: g < 0 for k in [{k[p.butterfly_g(k, *bad) < 0].min():.2f}, {k[p.butterfly_g(k, *bad) < 0].max():.2f}]")

## 4. Calendar arbitrage and interpolation

Across expiries, total variance at a fixed `k` must **not decrease** with `T` (a longer option can't be worth less). Return every `(T_short, T_long, k)` on the grid where `w` falls from one expiry to the next (`w_long < w_short`).

In [ ]:
surface = {30 / 365: p.TRUE_SVI,
           60 / 365: (0.0030, 0.050, -0.65, 0.03, 0.10),
           90 / 365: (0.0044, 0.030, -0.60, 0.02, 0.10)}        # the 90-day smile holds too little total variance
k_grid = np.round(np.linspace(-0.3, 0.2, 11), 3)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def calendar_violations(k_grid, params_by_T):
    Ts = sorted(params_by_T)
    out = []
    for t1, t2 in zip(Ts, Ts[1:]):
        w1 = p.svi_total_var(k_grid, *params_by_T[t1])
        w2 = p.svi_total_var(k_grid, *params_by_T[t2])
        out += ...                                # ✍️ [(t1, t2, float(k)) for each k where w2 < w1 − 1e-12]
    return out

mine = p.attempt(calendar_violations, k_grid, surface)
mine = p.check("calendar_violations", mine, p.calendar_violations(k_grid, surface))
[(round(a * 365), round(b * 365), k) for a, b, k in mine]

Between listed expiries, interpolate **total variance** linearly in `T` at fixed `k`, then convert back: `σ = √(w/T)`. Interpolating the vols themselves can create calendar arbitrage even when both expiries are clean.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def interp_iv(k, T, T1, params1, T2, params2):
    w1, w2 = p.svi_total_var(k, *params1), p.svi_total_var(k, *params2)
    w = ...                                       # ✍️ linear in T between (T1, w1) and (T2, w2)
    return np.sqrt(w / T)

T1, T2 = 30 / 365, 60 / 365
cases = [(0.0, 45 / 365), (-0.1, 40 / 365), (0.05, 55 / 365)]
mine = [p.attempt(interp_iv, k, t, T1, surface[T1], T2, surface[T2]) for k, t in cases]
expected = [np.sqrt((p.svi_total_var(k, *surface[T1]) + (p.svi_total_var(k, *surface[T2]) - p.svi_total_var(k, *surface[T1])) * (t - T1) / (T2 - T1)) / t)
            for k, t in cases]
mine = p.check("interp_iv", mine, expected)
[f"{x:.2%}" for x in mine]

## 5. SABR: a model with meaning in its parameters

SABR models the forward and its volatility as two correlated processes. Its four parameters read like a trader's view: `α` the level, `β` the backbone (how vol moves with the forward), `ρ` the spot–vol correlation (skew), `ν` the vol of vol (curvature). Hagan's formula gives the implied vol directly (`p.sabr_vol`).

In [ ]:
Fq, Ks = 100.0, np.linspace(70, 130, 121)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for rho_ in (-0.6, -0.3, 0.0, 0.3):
    axes[0].plot(Ks, p.sabr_vol(Fq, Ks, 1.0, 0.2, 1.0, rho_, 0.5) * 100, label=f"ρ = {rho_}")
for nu in (0.1, 0.4, 0.8):
    axes[1].plot(Ks, p.sabr_vol(Fq, Ks, 1.0, 0.2, 1.0, -0.3, nu) * 100, label=f"ν = {nu}")
axes[0].set_title("ρ tilts the smile (skew)"); axes[1].set_title("ν bends it (curvature)")
for ax in axes:
    ax.set_xlabel("strike (F = 100)"); ax.legend()
axes[0].set_ylabel("IV, %"); plt.tight_layout(); plt.show()

## Wrap-up

* Fit smiles in total variance with bounds, then **check** them: butterfly within an expiry, calendar across expiries.
* Interpolate in total variance, not in vol.
* Graded version: `labs/part06/week22_surface_portfolio` (SVI recovery, Durrleman and calendar checks, SABR) and the Clinic W2 surface snapshot.